# Customer Segmentation Using K-Means Clustering

## 1. Problem Statement
Businesses generate massive amounts of transactional data. Without a structured approach to understand customer behavior, marketing efforts become generic and inefficient. Customer segmentation is required to tailor strategies, maximize ROI, and increase retention.

## 2. Objective
Our goal is to utilize **Unsupervised Machine Learning** to automatically discover hidden patterns and distinct customer groups within the dataset, without relying on predefined labels. This allows the algorithm to mathematically determine the optimal segments.

## 3. Dataset
We are using the `Online Retail.xlsx` dataset, containing transactional records for a UK-based online retailer. Let's load the data.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append('../src')
from preprocessing import load_data, clean_data
from feature_engineering import compute_rfm_features, compute_additional_features
from clustering import preprocess_for_clustering, find_optimal_k, train_final_model
from evaluation import evaluate_clusters, assign_segment_names, map_segments_to_data
from visualization import plot_elbow_and_silhouette, plot_rfm_distributions, plot_pca_clusters
from inference import predict_new_customer, get_human_readable_explanation
from analytics import analyze_temporal_sales, market_basket_analysis
from recommendations import generate_segment_recommendations

# Load Data
df = load_data('../data/Online Retail.xlsx')
display(df.head())

## 4. Data Preprocessing
Real-world data is noisy. We clean the data to prevent our model from learning incorrect patterns.

- We drop rows with missing `CustomerID`.
- We remove duplicate rows.
- We filter out cancelled transactions (InvoiceNo starts with 'C').
- We ensure Quantity and UnitPrice are strictly positive.

In [ ]:
clean_df = clean_data(df)
display(clean_df.describe())

## 5. RFM Feature Engineering
We convert transactional rows into customer-level behavioral features known as RFM:
- **Recency**: Days since the customer's last purchase.
- **Frequency**: Total number of unique purchases.
- **Monetary**: Total amount spent.

We also engineer additional features like Average Order Value, Churn Risk %, 12M Projected CLV $, and Engagement Index.

In [ ]:
rfm_df = compute_rfm_features(clean_df)
rfm_df = compute_additional_features(clean_df, rfm_df)
display(rfm_df.head())

## 6. Why K-Means?
K-Means is highly efficient for customer segmentation. It partitions data into K distinct non-overlapping subsets by finding cluster centroids that minimize the variance within each cluster.

## 7. Why Scaling?
K-Means relies on distance calculations (Euclidean distance). Since 'Monetary' values are much larger than 'Frequency' values, Monetary would incorrectly dominate the distance calculations. `StandardScaler` ensures all features contribute equally.

## 8. Why Log Transformation?
Customer purchasing data is highly right-skewed (few customers spend massive amounts). Log transformation normalizes the distribution, allowing K-Means to identify spherical clusters more easily.

In [ ]:
X_scaled_df, scaler = preprocess_for_clustering(rfm_df, features=['Recency', 'Frequency', 'Monetary'])
plot_rfm_distributions(rfm_df, output_dir='../outputs/plots')
from IPython.display import Image
display(Image(filename='../outputs/plots/rfm_distributions.png'))

## 9. Choosing K
We use two mathematical methods to find K:
1. **Elbow Method (Inertia/WCSS)**: Finds the point of diminishing returns in minimizing within-cluster variance.
2. **Silhouette Score**: Measures how similar an object is to its own cluster compared to other clusters (-1 to 1).

In [ ]:
wcss, silhouette_scores = find_optimal_k(X_scaled_df, max_k=10)
plot_elbow_and_silhouette(wcss, silhouette_scores, output_dir='../outputs/plots')
display(Image(filename='../outputs/plots/optimal_k_evaluation.png'))

### K-Value Justification
While K=2 may have the highest absolute Silhouette score, separating a business into just two segments lacks actionable granularity. The score stabilizes and slightly rises at K=4, providing the optimal trade-off between mathematical cohesion and business interpretability.

## 10. K-Means Model Execution
The model algorithm works as follows:
1. Select K initial centroids randomly.
2. Assign each customer to the nearest centroid.
3. Recalculate centroids based on assigned customers.
4. Repeat until convergence (WCSS is minimized).

In [ ]:
rfm_df, kmeans_model = train_final_model(rfm_df, X_scaled_df, k=4, scaler=scaler, model_dir='../models')
display(rfm_df.head())

## 11. Cluster Results
We evaluate the median behaviors of each mathematical cluster to dynamically map them to human-readable segments.

In [ ]:
cluster_summary = evaluate_clusters(rfm_df)
cluster_summary, segment_map = assign_segment_names(cluster_summary)
rfm_df = map_segments_to_data(rfm_df, segment_map)

os.makedirs('../outputs', exist_ok=True)
rfm_df.to_csv('../outputs/customer_segments.csv', index=False)
cluster_summary.to_csv('../outputs/cluster_summary.csv', index=False)
display(cluster_summary)

## 12. PCA Visualization
We use Principal Component Analysis strictly to reduce our scaled 3D feature space to 2D for visual plotting. The actual model was trained on the full original space.

In [ ]:
plot_pca_clusters(X_scaled_df, rfm_df['Cluster'], segment_map, output_dir='../outputs/plots')
display(Image(filename='../outputs/plots/pca_clusters_2d.png'))

## 13. Market Basket Analysis & Temporal Sales Insights
We perform co-occurrence association analysis to discover items frequently bought together and extract peak sales hours/days.

In [ ]:
temporal_data = analyze_temporal_sales(clean_df)
basket_df = market_basket_analysis(clean_df, min_support=0.003, top_n=10)
display(basket_df.head())

## 14. New Customer Prediction & Calibrated Inference
Using the `inference.py` pipeline, we apply identical transformation and scaling to predict segment placement, segment probability, centroid distances, Churn Risk, CLV, and persona classification.

In [ ]:
c_id, dists, probs, scaled, ai_metrics = predict_new_customer(recency=15, frequency=12, monetary=400, model_dir='../models')
seg_name, dist_dict, prob_dict, explanation = get_human_readable_explanation(c_id, dists, probs, cluster_summary)
print(f"Predicted Segment: {seg_name} (Probability: {prob_dict[seg_name]}%)\n")
print(f"Customer Persona: {ai_metrics['micro_persona']}")
print(f"Churn Risk: {ai_metrics['churn_risk_pct']}%")
print(f"Projected 12M CLV: ${ai_metrics['projected_clv']:,.2f}")
print(f"Engagement Index: {ai_metrics['engagement_index']} / 100\n")
print("Centroid Distances & Probabilities:")
for k, v in dist_dict.items():
    print(f"- {k}: Distance = {v}, Probability = {prob_dict[k]}%")
print(f"\nExplanation: {explanation}")

## 15. Conclusion & Action Plan
We successfully implemented an end-to-end Unsupervised Learning & Analytics pipeline that segments customers, discovers product affinity bundles, and recommends personalized next-best actions.